### Quant Data Engine — Code Walkthrough

This notebook walks through how the engine works, piece by piece. We start with the
Binance loader.

One thing to know upfront: both the Binance and Kraken loaders make their HTTP requests
through a shared retry helper (`get_with_retry` in `http.py`). It retries automatically on
temporary failures — rate limits (429) and server errors (5xx) — using exponential backoff,
and fails fast on permanent errors like a bad symbol. It exists as one shared function so the
retry logic isn't copy-pasted into every loader.

In [ ]:
# To make sure the created documents are at the project root level
import os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
print(os.getcwd())

In [ ]:
# Make sure any update to imported module reloads automatically
%load_ext autoreload
%autoreload 2

##### The binance loader

`load_binance_ohlcv` fetches historical OHLCV (open, high, low, close, volume) candles
from Binance's REST API and returns them as a clean, standardized DataFrame. Here is what
happens under the hood, in order.

**1. Date conversion.** The caller passes a human-readable date like `"2020-01-01"`, but
Binance only accepts timestamps as epoch milliseconds. The loader interprets the string as
UTC, converts it to epoch seconds, and multiplies by 1000 to get milliseconds.

**2. The request.** A `params` dictionary is built using Binance's exact parameter names
(`symbol`, `interval`, `startTime`, `endTime`, `limit`). The request goes through the shared
`get_with_retry` helper, and the JSON response is parsed into Python objects — a list of
candles, where each candle is a list of positional values.

**3. Pagination.** Binance returns at most 1000 candles per request. To cover longer ranges,
the loader loops: it fetches a batch, appends it to a running list, then advances the start
time to one millisecond past the last candle received so the next request continues where the
previous one stopped. The `+1` prevents the boundary candle from being fetched twice. The loop
stops when a batch comes back empty or smaller than 1000 — both meaning there is no more data.

**4. Building the DataFrame.** The accumulated list of lists becomes a table. Column names are
assigned manually, in the order Binance documents, because the response carries no labels —
the values are identified purely by position.

**5. Cleaning.** Prices and volumes arrive as strings (financial APIs send numbers as text to
preserve exact precision), so the numeric columns are converted with `pd.to_numeric`. The
open-time column is converted to a UTC-aware DatetimeIndex named `date`, which turns the table
into a proper time series — enabling date slicing, cross-asset alignment, resampling, and the
quality checks.

**6. Trimming.** Finally the DataFrame is reduced to the five OHLCV columns. This enforces the
common schema shared by all three loaders, so that data from Binance, Kraken, and Yahoo Finance
is structurally identical and interchangeable downstream. Binance's extra fields (taker buy
volumes, trade count) are dropped here, though they would be valuable for future order-flow
analysis.

In [ ]:
# The binance loader in action
from qde.loaders.binance_loader import load_binance_ohlcv

# called directly using its own symbol format ("BTCUSDT")
df = load_binance_ohlcv("BTCUSDT", start="2020-01-01")
df.head()


#### The yfinance loader

`load_yfinance_ohlcv` fetches OHLCV data from Yahoo Finance. Unlike Binance and Kraken,
yfinance is a convenience library, not a raw REST API — so the fetching is simple, but the
output needs more cleaning because its shape is less predictable.

**1. The fetch.** A single call to `yf.download` returns the whole date range at once — no
pagination loop, no manual requests, no JSON parsing; the library handles all of that
internally. `auto_adjust=True` requests prices adjusted for splits and dividends, so the
series reflects true economic returns rather than showing artificial jumps on corporate-action
dates — the correct choice for comparing assets over time.

**2. The MultiIndex quirk.** yfinance sometimes returns columns as a two-level MultiIndex,
where level 0 is the field (open, high, close...) and level 1 is the ticker. Since this loader
handles one symbol at a time, that second level is redundant. The loader checks
`isinstance(data.columns, pd.MultiIndex)` first — because yfinance's output shape isn't
guaranteed — and only then drops the redundant level with `droplevel(1, axis="columns")`.
Detect the actual state, then fix it.

**3. The timezone quirk.** yfinance is also inconsistent about timezones: sometimes the index
is naive (no timezone), sometimes already aware. The loader checks `data.index.tz`. If naive,
it uses `tz_localize("UTC")` to attach UTC. If aware, it uses `tz_convert("UTC")` to shift into
UTC. The distinction matters — each method only works on one state and errors on the other — so
detecting the state first is essential.

**4. Cleaning.** From here the steps mirror the Binance loader: lowercase and order the columns,
confirm the UTC DatetimeIndex named `date`, and trim to the five OHLCV
columns — the same common schema every loader produces.

The contrast with Binance is worth noting: Binance gives raw positional data that is awkward but
predictable; yfinance gives a friendlier interface but less predictable output. Both are handled
by the same defensive principle — inspect the actual data, never assume its shape.

In [ ]:
# The yfinance loader in action
from qde.loaders.yfinance_loader import load_yfinance_ohlcv

# called directly using its own symbol format. For bitcoin, it would be ("BTC-USD").
df = load_yfinance_ohlcv("SPY", start="2020-01-01")
df.head()

#### The Kraken loader

`load_kraken_ohlcv` follows the same overall pattern as the Binance loader — request, clean,
return — but Kraken has several conventions of its own that the loader has to bridge.

**1. Interval in minutes.** Kraken expects the bar size as a number of minutes, not a string.
An `interval_map` translates the project's standard format (`"1d"`, `"1h"`) into Kraken's
minutes (`1440`, `60`). The lookup uses `.get()` so an unsupported interval returns `None` and
raises a clear error, rather than a cryptic `KeyError`.

**2. Timestamps in seconds.** Unlike Binance (milliseconds), Kraken uses epoch *seconds*. So
the start conversion skips the `×1000`, and the timestamp-to-index step later uses `unit="s"`.
Getting this wrong would shift every date by a factor of 1000.

**3. The nested, renamed response.** Kraken wraps its response: `data["result"]` is a dictionary
with two keys — the candle data and a `last` cursor. Critically, Kraken renames the pair (you
ask for `XBTUSD`, it returns data under `XXBTZUSD`), so the key can't be hardcoded. The loader
finds it by exclusion — `[k for k in result if k != "last"][0]` — "the one key that isn't the
cursor." The `[0]` unwraps the single-item list the comprehension produces.

**4. Cursor pagination.** Where Binance made you compute the next start from the last candle's
timestamp, Kraken hands you the next cursor directly as `result["last"]`. The loop advances using
that cursor and stops when it stops moving (no more data). In practice this is largely moot:
Kraken's public OHLC endpoint only serves roughly the most recent 720 candles per interval
regardless of the requested start, so deep history isn't available through this endpoint — that
would require Kraken's paid data service.

**5. Cleaning.** From here the steps mirror the other loaders: build the DataFrame, deduplicate
on the index (pagination can repeat a candle at batch boundaries), convert numerics, set the
UTC `date` index (`unit="s"`), and trim to the five OHLCV columns.

In [ ]:
# The kraken loader in action
from qde.loaders.kraken_loader import load_kraken_ohlcv

# Like the others, called directly using its own symbol format ("XBTUSD")
df = load_kraken_ohlcv("XBTUSD", start="2024-01-01")
df.head()

#### The unified loader and symbol mapping

The three loaders each speak their source's native symbol format — Binance wants `BTCUSDT`,
yfinance wants `BTC-USD`, Kraken wants `XBTUSD`. Forcing the user to remember which format goes
with which source would leak all that complexity back out. The unified layer hides it.

###### SYMBOL_MAP (symbols.py)

A nested dictionary. The outer key is the *source*; the inner key is the *canonical symbol*; the
inner value is that source's *native symbol*:

    SYMBOL_MAP = {
        "yfinance": {"BTCUSDT": "BTC-USD", "SPY": "SPY"},
        "binance":  {"BTCUSDT": "BTCUSDT"},
        "kraken":   {"BTCUSDT": "XBTUSD"},
    }

The canonical format follows Binance's convention, chosen because the project is crypto-first and
that format is the common crypto convention. The map serves two purposes: it **translates** one
canonical symbol into each source's native format, and it acts as a **validity registry** —
a symbol-source combination is only valid if it appears in the map. `BTCUSDT` works on all three
sources; `SPY` appears only under yfinance, so requesting it from Binance is correctly rejected.

##### load_ohlcv (loaders/__init__.py)

This is the single entry point. It does two jobs:

**Translation.** `SYMBOL_MAP.get(source, {}).get(symbol)` walks the two dict levels — first the
source's inner dict, then the symbol within it — returning the native symbol. The `{}` default on
the first lookup is a defensive touch: if the source is unknown, the first `.get()` returns an
empty dict instead of `None`, so the second `.get()` returns `None` cleanly rather than crashing.
A single `if mapped is None` guard then catches both unknown sources and unknown symbols with one
clear error.

**Routing.** An if/elif on `source` calls the matching loader, passing `mapped` — the *translated*
symbol — so each loader receives the format it expects. An unsupported source raises a clear error
listing the valid options.

The result is the project's core abstraction: the caller uses one symbol and one function, and the
per-source differences in symbol format and API behaviour stay hidden behind this layer.

In [ ]:
from qde.loaders import load_ohlcv

# The same canonical symbol, across all three sources. The user never touches
# BTC-USD or XBTUSD — load_ohlcv translates internally.
btc_binance = load_ohlcv("BTCUSDT", start="2024-01-01", source="binance")
btc_kraken  = load_ohlcv("BTCUSDT", start="2024-01-01", source="kraken")
btc_yahoo   = load_ohlcv("BTCUSDT", start="2024-01-01", source="yfinance")

print("All three return identical structure:")
print(btc_binance.columns.tolist())

# And the validity registry in action — SPY isn't available on Binance:
try:
    load_ohlcv("SPY", start="2024-01-01", source="binance")
except ValueError as e:
    print(f"Correctly rejected: {e}")

#### Storage

Fetched data lives only in memory. The storage layer persists it as Parquet files on disk, so
data is fetched once and reused — no repeated API calls.

#### _ohlcv_path (the shared path helper)

A small internal helper that builds the file path for a given symbol, source, and interval —
returning something like `data/ohlcv/BTCUSDT_binance_1d.parquet`. It exists so the naming scheme
lives in exactly one place. Every storage function (`save`, `load`, `update`) calls it instead of
constructing paths itself, which means they can never disagree about where a file should be: save
writes to a path and load reads from the identical path because both compute it the same way. If
the convention ever changes, it changes here once. The `_` prefix marks it as internal — not part
of the public API. This is the same DRY principle behind the shared `get_with_retry` HTTP helper.

In [ ]:
from qde.storage import _ohlcv_path
_ohlcv_path("BTCUSDT", "binance", "1d", "data")

#### save_ohlcv

Fetches data through the unified `load_ohlcv` and persists it to disk as Parquet. The steps:
call the loader to get a clean DataFrame, build the destination path via `_ohlcv_path`, ensure the
directory exists with `mkdir(parents=True, exist_ok=True)` — `parents=True` creates any missing
parent folders, `exist_ok=True` means running it twice doesn't crash — write the DataFrame with
`to_parquet`, and return the path as a string. Returning the path lets the caller log or chain on
where the file landed, and makes the function easy to test.

In [ ]:
from qde.storage import save_ohlcv
path = save_ohlcv("BTCUSDT", source="binance", start="2024-01-01")
print(f"Saved to: {path}")

#### load_ohlcv_local

The mirror of `save_ohlcv`: it reads a previously saved Parquet file back into a DataFrame with no
API call — instant and fully offline. It builds the path with the same `_ohlcv_path` helper (which
is why save and load always agree on location), then checks the file exists. If not, it raises
`FileNotFoundError` — the precise exception for a missing file, clearer than a generic error and
caught before `pd.read_parquet` can throw something cryptic. If the file is there, it reads and
returns it. The UTC `date` index is preserved automatically, since Parquet stores index metadata.

In [ ]:
from qde.storage import load_ohlcv_local
df = load_ohlcv_local("BTCUSDT", source="binance")
df.tail()

#### update_ohlcv (incremental refresh)

Keeps a stored dataset current while fetching only what's missing — never re-downloading history
already on disk. The steps: read the existing file, find the most recent stored date with
`index.max()`, and compute the next day (`+ pd.Timedelta(days=1)`, then `.date()` to a string the
loader accepts). It then fetches from that day onward via the unified loader, wrapped in a
`try/except`: if the source has no new data, the loader raises `ValueError`, which is caught and
treated as the normal "already up to date" case rather than a crash. When new data does arrive, it
is concatenated onto the existing data and deduplicated with `keep="last"`, so if a boundary day
appears in both, the newer (more complete) version wins. The combined result is written back to the
same Parquet file. This is what makes a daily refresh cheap — a handful of new rows instead of
years of history.

In [ ]:
from qde.storage import update_ohlcv
update_ohlcv("BTCUSDT", source="binance")

#### query (SQL via DuckDB)

Exposes all stored data to real SQL. It opens a transient in-memory DuckDB engine — there is no
standing database; the Parquet files *are* the storage, and DuckDB is an engine that reads them
directly from disk. The function scans the data folder and registers every `.parquet` file as a
SQL view named after the file (so `BTCUSDT_binance_1d.parquet` becomes the table
`BTCUSDT_binance_1d`), then runs the caller's SQL and returns a DataFrame. The key property is
automatic registration: table names are never hardcoded — they're discovered by scanning the
folder — so a newly saved dataset becomes queryable by name with zero code changes. This is the
same folder-scanning pattern used by the daily update and quality summary.

In [ ]:
from qde.storage import query
query("SELECT date, close FROM BTCUSDT_binance_1d ORDER BY close DESC LIMIT 5")

#### Quality layer

Four automated checks validate every stored dataset, plus a runner that summarizes them and a
builder that writes the results to CSV for the Power BI dashboard. Each check is a pure function:
it takes a DataFrame and returns what it found, knowing nothing about where the data came from.

#### check_gaps
Detects missing trading days. Both branches build the dates that *should* exist, then subtract the
dates actually present to find what's missing. For **crypto**, every day is a trading day, so the
expected set is a full daily range. For **equities**, weekends and holidays are legitimate closures
— so it uses the NYSE calendar (via `pandas_market_calendars`) to ask which days the market was
actually open, and only a genuinely absent trading day counts as a gap. The NYSE schedule comes back
timezone-naive, so it's localized to UTC before comparing against the UTC index.

In [ ]:
from qde.storage import load_ohlcv_local
from qde.quality import check_gaps

btc = load_ohlcv_local("BTCUSDT", source="binance")
spy = load_ohlcv_local("SPY", source="yfinance")
print("BTC gaps:", len(check_gaps(btc, calendar="crypto")))
print("SPY gaps:", len(check_gaps(spy, calendar="equity")))

#### check_duplicates
Returns any rows whose timestamp appears more than once. Duplicates shouldn't exist, but could be
introduced by a faulty incremental update, so the check guards against silent corruption.
`keep=False` marks *all* copies of a duplicated timestamp, so every offending row is returned for
inspection, not just the repeats.

In [ ]:
from qde.quality import check_duplicates
check_duplicates(btc)

#### check_nulls
Returns the count of null values per column. Market data should have no empty fields, so any
non-zero count signals a problem with the source or the cleaning.

In [ ]:
from qde.quality import check_nulls
check_nulls(btc)

#### check_price_sanity
Returns any rows that are physically impossible: a price at or below zero in any OHLC column, or a
high below the low. These conditions can't occur in real market data, so their presence indicates
corruption. The conditions are combined with boolean OR so a row failing *any* one is flagged.

In [ ]:
from qde.quality import check_price_sanity
check_price_sanity(btc)

#### run_quality_report
Runs all four checks on a single DataFrame and prints a readable summary, returning a boolean —
`True` if everything is clean. This is the human-facing view for one dataset.

In [ ]:
from qde.quality import run_quality_report
run_quality_report(btc, name="BTCUSDT_binance_1d", calendar="crypto")

#### build_quality_summary
The machine-facing view. It scans every Parquet file in the data folder, runs all four checks on
each, computes freshness (`days_stale`) and row counts, assembles one row per dataset into a
DataFrame, stamps it with a generation timestamp, and writes it to `quality_summary.csv` — the file
the Power BI dashboard reads. Same folder-scanning pattern as the query layer and daily update.

In [ ]:
from qde.quality import build_quality_summary
build_quality_summary()

#### Automation and tests

#### scripts/daily_update.py
Run nightly by Windows Task Scheduler. It scans the data folder, updates each stored dataset with
`update_ohlcv` (each wrapped in try/except so one failure doesn't abort the rest), then rebuilds the
quality summary CSV. Because it discovers datasets by scanning files, new symbols are picked up
automatically with no change to the script.

#### tests/
A pytest suite verifying each module's contract. Loader tests confirm a valid call returns a
non-empty DataFrame with the exact expected columns, and that an invalid symbol raises. Storage
tests use the `tmp_path` fixture — a temporary directory pytest creates and deletes — passed as
`base_dir`, so tests never touch real data. `pytest.raises` asserts that error guards actually fire
on bad input. The suite is what lets the system be changed with confidence: if a refactor breaks a
contract, a test catches it.

In [ ]:
# Run from a terminal, not the notebook:
# pytest